In [1]:
from demoparser2 import DemoParser
import pandas as pd
from pathlib import Path
import numpy as np

In [4]:
#BASE_DIR = Path(__file__).resolve().parent
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'

files = [str(p) for p in list(DATA_DIR.rglob('*.dem'))]

In [ ]:
PLAYER_PROPS = [
    "team_name",
    "tick",
    "total_rounds_played",
    "player_name",
    "team_num",
    "health",
    "X", "Y", "Z",
    "velocity_X", "velocity_Y", "velocity_Z",
    "yaw", "pitch",
    "is_match_started",
    "is_alive",
    "active_weapon_name",
    "inventory",
    "has"
    "current_equip_value",
    "team_rounds_total",
    "is_match_started"
]

KEEP_COLS = [
    'tick',
    'steamid',
    'name',
    'tick',
    'round_num',
    'health',
    'X', 'Y', 'Z',
    'velocity_X', 'velocity_Y', 'velocity_Z',
    'yaw', 'pitch',
    'team_name',
    'inventory',
    'is_alive',
    'is_planted',
    'bomb_site',
]

EVENT_PROPS = [
    'round_announce_match_start',
    'cs_round_final_beep',
    'cs_win_panel_match',
    'bomb_planted',
    'bomb_exploded',
    'bomb_defused',
    'grenade_thrown',
    'player_death',
    'player_hurt',
    'round_start',
    'round_end',
]

DEATH_COLS = [
    'tick',
    'user_steamid',
    'dmg_health', 
    'weapon',
    'attacker_steamid', 
    'attackerblind', 
    'attackerinair', 
    'headshot', 
    'hitgroup', 
    'noscope', 
    'penetrated', 
    'thrusmoke', 
    'assistedflash', 
    'assister_steamid',
]

HURT_COLS = [
    'tick',
    'user_steamid',
    'health',
    'dmg_health',
    'weapon',
    'attacker_steamid',
    'hitgroup',
]

NADE_COLS = [
    'tick',
    'user_steamid',
    'weapon',
]

def parse_demo(path):
    parser = DemoParser(path)

    players = parser.parse_ticks(PLAYER_PROPS)
    events = parser.parse_events(EVENT_PROPS, player=["last_place_name"])
    info = parser.parse_header()

    return players, events, info

def parse_file(file):
    players, events, info = parse_demo(file)
    players['steamid'] = players['steamid'].astype(str)
    players = players[players['is_match_started'] == True]
    players = players.copy()
    map_name = info.get('map_name')

    #EVENTS INTO DICT
    frames = {frame[0]:frame[1] for frame in events}
    
    #EXTRACT EVENTS
    match_start = frames.get('round_announce_match_start')
    if match_start is not None:
        match_start = match_start.loc[0, 'tick']
    round_starts = frames.get('cs_round_final_beep')
    match_end = frames.get('cs_win_panel_match')
    if match_end is not None:
        match_end = match_end.loc[0, 'tick']
    rounds = frames.get('round_start')
    if rounds is not None:
        rounds = rounds.loc[rounds.groupby('round')['tick'].idxmax()]
    round_ends = frames.get('round_end')
    if round_ends is not None:
        round_ends = round_ends[(round_ends['reason'].notna()) & (round_ends['tick'] != 0) & (round_ends['winner'].notna())]
        round_ends['round'] = np.arange(1, len(round_ends) + 1)
    deaths = frames.get('player_death')
    if deaths is not None:
        deaths = deaths[DEATH_COLS]
    hurt = frames.get('player_hurt')
    if hurt is not None:
        hurt = hurt[HURT_COLS]
    nades = frames.get('grenade_thrown')
    if nades is not None:
        nades = nades[NADE_COLS]
    planted = frames.get('bomb_planted')
    defused = frames.get('bomb_defused')
    explode = frames.get('bomb_exploded')

    #START AND END OF ROUND
    round_numbers = (players[players['tick'].isin(round_starts['tick'])][['tick', 'total_rounds_played']]
                        .rename(columns={'tick':'start_tick', 'total_rounds_played':'start_total_rounds_played'}))
    
    #MATCH BOUNDARIES
    if match_end:
        players = players[players['tick'] <= match_end]
    if match_start:
        players = players[players['tick'] >= match_start]
    players = players.copy()

    #START TICK MERGE
    players = pd.merge_asof(players, round_numbers, left_on='tick', right_on='start_tick', direction='backward')
    in_round = players['total_rounds_played'] == players['start_total_rounds_played']
    shifted_in_round = in_round.shift(1).fillna(False)
    players = players[in_round | shifted_in_round]
    players['round_num'] = players['total_rounds_played'] + 1

    #BOMB INFO
    if planted is not None:
        bomb_df = players.merge(planted.drop(columns='user_name'), left_on=['tick', 'steamid'], right_on=['tick', 'user_steamid'], how='left').rename(columns={'user_last_place_name':'bomb_site'})
        players['bomb_planted'] = bomb_df['site'].notna().astype(int)

        players['is_planted'] = np.where(players['bomb_planted']==1, True, pd.NA)
        players['is_planted'] = players.groupby('round_num')['is_planted'].ffill()
        players['is_planted'] = players['is_planted'].astype('boolean').fillna(False)

        players['bomb_site'] = np.where(bomb_df['bomb_site'].notna(), bomb_df['bomb_site'], pd.NA)
        players['bomb_site'] = players.groupby('round_num')['bomb_site'].ffill()

    #GET TICKS WHERE EVENT OCCURS
    important_ticks = pd.concat([
        round_starts['tick'],
        round_ends['tick'],
        deaths['tick'],
        hurt['tick'],
        nades['tick'],
        planted['tick'],
        defused['tick'],
        explode['tick']
    ])

    #CREATE MASKS FOR EVENTS
    important_ticks = important_ticks[important_ticks.isin(players['tick'])]
    event_mask = players['tick'].isin(important_ticks)

    #CREATE MASKS FOR 8TH TICKS
    players['round_tick'] = players['tick']-players['start_tick']
    eighth_mask = (players['round_tick'] % 8 == 0)

    players = players[event_mask | eighth_mask]
    players = players[KEEP_COLS]

    return {
        'map_name': map_name,
        'ticks': players,
        'rounds': rounds,
        'ends': round_ends,
        'deaths': deaths,
        'hurt': hurt,
        'nades': nades,
        'plant': planted,
        'defuse': defused,
        'explode': explode,
    }

In [ ]:
players = parse_file(files[0])

In [ ]:
players[(players['steamid']=='76561197989430253') & (players['total_rounds_played']==1)][['X','Y','Z','tick','team_rounds_total']]

In [ ]:
players[players['is_planted'] == True]

In [ ]:
players

In [ ]:
players = players.drop(columns=['is_match_started', ''])


In [ ]:
players[['']]

In [ ]:
KEEP_COLS = [
    'tick',
    'steamid',
    'name',
    'tick',
    'round_num',
    'health',
    'X', 'Y', 'Z',
    'velocity_X', 'velocity_Y',
    'yaw', 'pitch',
    'team_name',
    'inventory',
    'is_alive',
    'is_planted',
    'bomb_site',
]

In [ ]:
files

In [ ]:
players, events, info = parse_demo(files[0])

In [ ]:
test = {k[0]: k[1] for k in events}

In [ ]:
test['round_end']

In [ ]:
players[players['tick'] == 189630]

In [ ]:
test['player_death']

In [ ]:
players['start_tick']

In [ ]:
test['rounds']

In [5]:
parser = DemoParser(files[0])

In [12]:
parser.parse_event('player_hurt').dtypes

armor                int32
attacker_name       object
attacker_steamid    object
dmg_armor            int32
dmg_health           int32
health               int32
hitgroup            object
tick                 int32
user_name           object
user_steamid        object
weapon              object
dtype: object

In [14]:
parser.parse_event('player_death')['penetrated'].value_counts()

penetrated
0    137
1      4
Name: count, dtype: int64

In [ ]:
inv = test['inventory'].explode()
uni = inv.unique()
print(uni)

In [ ]:
inv[inv=='Zeus x27']

In [ ]:
len(players['ticks']['steamid'].unique())

In [ ]:
test.iloc[437154]

In [ ]:
parser.parse_event('bomb_planted')

In [ ]:
parser.parse_event('bomb_exploded')

In [ ]:
parser.parse_event('bomb_defused')